# Play chess against ChessGPT

This notebook lets you play an interactive game against ChessGPT (432M params).

- **Click on a piece** to select it — legal moves will be highlighted
- **Click on a highlighted square** to play the move
- For promotions, a popup lets you choose the piece

In [ ]:
import torch
import chess

from chessgpt import resolve_device
from chessgpt.generate import (
    load_checkpoint,
    sample_next_token,
    build_legal_mask,
)
from chessgpt.board_widget import ChessBoardWidget

# Load model from the HuggingFace Hub
device = resolve_device("auto")
model, tokenizer, config = load_checkpoint(device=device)
print(f"Model loaded on {device} — {sum(p.numel() for p in model.parameters()):,} params")
print(f"Architecture: d_model={config.d_model}, n_layers={config.n_layers}, n_heads={config.n_heads}")

In [2]:
# --- Game settings ---

PLAYER_COLOR = chess.WHITE   # chess.WHITE or chess.BLACK

# Sampling
TEMPERATURE = 0.4
TOP_K = 40
TOP_P = 0.95
GREEDY = False               # True = always pick the model's top move

In [3]:
def model_play_move(board, history):
    """Have the model play a move and return the UCI string."""
    ids = [tokenizer.BOS_ID] + [tokenizer.move_to_id[m] for m in history]

    # Crop to max context length
    max_ctx = model.config.max_seq_len
    if len(ids) > max_ctx:
        ids = ids[-max_ctx:]

    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    legal_mask = build_legal_mask(board, tokenizer, device, allow_eos=False)

    tid = sample_next_token(
        model=model,
        input_ids=input_ids,
        tokenizer=tokenizer,
        legal_mask=legal_mask,
        temperature=TEMPERATURE,
        top_k=TOP_K,
        top_p=TOP_P,
        greedy=GREEDY,
    )

    move_uci = tokenizer.id_to_move.get(tid, "")
    if not move_uci or chess.Move.from_uci(move_uci) not in board.legal_moves:
        import random
        mv = random.choice(list(board.legal_moves))
        move_uci = mv.uci()

    return move_uci


def format_history(history_san):
    """Format move history as HTML (1. e4 e5 2. Nf3 ...)."""
    parts = []
    for i in range(0, len(history_san), 2):
        move_num = i // 2 + 1
        white = history_san[i]
        black = history_san[i + 1] if i + 1 < len(history_san) else ""
        parts.append(f"<b>{move_num}.</b> {white} {black}")
    return "  ".join(parts)

In [4]:
# --- Game state ---
board = chess.Board()
history = []          # UCI move strings
history_san = []      # SAN move strings
move_number = 1

# --- Widget ---
color_str = "white" if PLAYER_COLOR == chess.WHITE else "black"
widget = ChessBoardWidget(player_color=color_str)
widget.update_position(board)
widget.set_status(f"Game started! You play {color_str}. Your turn." if PLAYER_COLOR == chess.WHITE
                  else f"Game started! You play {color_str}. ChessGPT is thinking...")


def refresh_widget(last_move=None):
    """Refresh widget display with current board state."""
    widget.update_position(board, last_move=last_move)
    widget.set_history(format_history(history_san))


def check_game_over():
    """Check if game is over and update status. Returns True if over."""
    if not board.is_game_over():
        return False
    outcome = board.outcome()
    result = board.result()
    if outcome.termination == chess.Termination.CHECKMATE:
        winner = "White" if outcome.winner == chess.WHITE else "Black"
        widget.set_status(f"Game over! {result} — Checkmate, {winner} wins!")
    elif outcome.termination == chess.Termination.STALEMATE:
        widget.set_status(f"Game over! {result} — Stalemate.")
    else:
        widget.set_status(f"Game over! {result} — Draw ({outcome.termination.name}).")
    # Clear legal moves so no more clicks are accepted
    widget.legal_moves = "[]"
    return True


def play_model_move():
    """Have the model play its move."""
    global move_number
    widget.set_status("ChessGPT is thinking...")

    move_uci = model_play_move(board, history)
    mv = chess.Move.from_uci(move_uci)
    san = board.san(mv)

    board.push(mv)
    history.append(move_uci)
    history_san.append(san)

    if board.turn == chess.WHITE:
        move_number += 1

    refresh_widget(last_move=mv)

    if check_game_over():
        return

    if board.is_check():
        widget.set_status(f"ChessGPT played {san}. Check! Your turn.")
    else:
        widget.set_status(f"ChessGPT played {san}. Your turn.")


def on_player_move(move_uci):
    """Callback when the player clicks a move on the board."""
    global move_number

    mv = chess.Move.from_uci(move_uci)
    if mv not in board.legal_moves:
        return

    san = board.san(mv)
    board.push(mv)
    history.append(move_uci)
    history_san.append(san)

    if board.turn == chess.WHITE:
        move_number += 1

    refresh_widget(last_move=mv)

    if check_game_over():
        return

    # Model's turn
    play_model_move()


widget.on_move(on_player_move)

# If player is black, model plays first
if PLAYER_COLOR == chess.BLACK:
    play_model_move()

display(widget)